# 技能2 · Day 2：Agent 编排架构 + LangGraph · 企业架构视角

**版本**：v5.0 学习材料包
**配套**：[notes.md](./notes.md)（讲义）｜ [data/README.md](./data/README.md)（真实库说明）｜ [reading.md](./reading.md)（深链阅读）
**核心命题**：如何用 LangGraph 的有状态图编排企业级多 Agent 营销工作流，实现顺序/循环/条件分支 + 人机协同审批（HITL）+ 状态持久化？
**v5.0 升级**：真实 LangGraph 库上机（非伪代码）+ TODO 填空 + 离线模拟 LLM fallback + 2026 前沿（A2A / Plan-Execute / 天道推演×多Agent仿真）


## 0. 环境准备

首次运行需安装依赖（取消注释执行一次）：
- `langgraph`：状态图编排框架（StateGraph / 条件边 / 检查点 / interrupt）
- `langchain-openai` + `langchain-core`：LLM 调用与消息类型
- `pydantic`：状态 schema 验证（备选方案）

无 `OPENAI_API_KEY` 时自动降级为**离线模拟 LLM**（返回固定营销文案），图可端到端跑通。


In [ ]:
# === 环境准备 ===
# 首次运行取消注释：!pip install langgraph langchain-openai langchain-core pydantic -q

import operator
from typing import TypedDict, Literal, Annotated
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.messages import HumanMessage, AIMessage, SystemMessage

# --- LLM 初始化 ---
# 优先使用真实 OpenAI API；无 API key 时自动降级为离线模拟 LLM
try:
    import os
    from langchain_openai import ChatOpenAI
    if os.environ.get("OPENAI_API_KEY"):
        llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7)
        LLM_MODE = "openai"
    else:
        raise ImportError("No OPENAI_API_KEY")
except Exception:
    LLM_MODE = "offline"

    class _MockResponse:
        """模拟 LLM 响应，兼容 LangChain .content 接口"""
        def __init__(self, content):
            self.content = content

    class OfflineMockLLM:
        """离线模拟 LLM：无 API key 时返回固定营销文案，使图端到端跑通。
        企业架构视角：每个分支模拟一个专用 Prompt+模型节点。"""
        _RESEARCH = (
            "【受众分析报告】\n"
            "目标人群：25-35岁都市白领女性，月收入1.5-3万，关注成分护肤\n"
            "竞品：修丽可(院线权威) / The Ordinary(极致性价比) / 珀莱雅(国货崛起)\n"
            "趋势：2026年烟酰胺趋向'温和浓度+复配功效'，小红书种草转化率最高\n"
            "建议角度：主打'温和不刺激+可见美白'，差异化'入门级功效护肤首选'"
        )
        _STRATEGY = (
            "【营销策略方案】\n"
            "核心主张：温和美白，每天可见\n"
            "差异化定位：5%黄金浓度+泛醇舒缓，入门级烟酰胺首选\n"
            "渠道组合：小红书40% + 抖音30% + 天猫30%\n"
            "信息层级：温和不刺激 -> 5%浓度 -> 28天可见效果\n"
            "预算分配：KOL种草40% + 信息流30% + 站内20% + 品牌号10%"
        )
        _COPY = (
            "【创意方案A】标题：每天2步，白成一道光\n"
            "正文：烟酰胺5%黄金浓度，搭配泛醇舒缓配方，28天看得见的美白。"
            "不刺激、不挑皮，入门级功效护肤第一选择。\n"
            "CTA：限时首发价129元，点击体验\n"
            "渠道：小红书\n\n"
            "【创意方案B】标题：温和美白，认真护肤\n"
            "正文：不是所有烟酰胺都叫温和。5%黄金浓度+泛醇，敏感肌也能安心用。\n"
            "CTA：抢先试用，享首发优惠\n"
            "渠道：抖音"
        )

        def invoke(self, messages):
            text = ""
            if isinstance(messages, str):
                text = messages
            elif isinstance(messages, list):
                text = " ".join(getattr(m, "content", str(m)) for m in messages)
            else:
                text = str(messages)
            if "受众" in text or "人群" in text or "分析" in text:
                return _MockResponse(self._RESEARCH)
            if "策略" in text or "主张" in text or "预算" in text:
                return _MockResponse(self._STRATEGY)
            return _MockResponse(self._COPY)

    llm = OfflineMockLLM()

print(f"LLM 模式: {LLM_MODE}")
print("LangGraph + langchain_core + pydantic 就绪")


## 1. 企业架构视角：为什么选 LangGraph 编排多 Agent

**与技能5 Day2 的区别（重要）**：
- 技能5 Day2 教 LangGraph **mechanics**（StateGraph/Node/Edge 怎么用）
- 本 Day 侧重**企业架构视角**：编排模式分类（顺序/并行/循环/条件）、多 Agent 协作拓扑（Supervisor/层级式）、人机协同治理节点（HITL 审批）、状态持久化与故障恢复

**LangGraph 的企业级价值**（vs LangChain 线性 Chain）：
| 维度 | LangChain Chain | LangGraph StateGraph |
|------|----------------|---------------------|
| 控制流 | 线性顺序 | 图结构（分支/循环/并行） |
| 状态管理 | 输入输出传递 | 全局 State + Checkpoint 持久化 |
| 人机交接 | 需额外工程 | 原生 `interrupt_before` |
| 生产就绪 | 弱 | 强（错误恢复/超时/重试） |

**四种编排模式**（本 Day 代码全部覆盖）：
1. **顺序（Sequential）**：research -> strategy -> copywriter
2. **条件分支（Conditional）**：approval -> publish OR revise
3. **循环（Loop）**：copywriter -> approval -> copywriter（带退出条件）
4. **人机协同（HITL）**：interrupt_before=["approval"] + update_state + resume

**营销映射**：多 Agent 协作完成营销活动--
研究Agent(分析受众) -> 策略Agent(制定方案, Plan-Execute) -> 写作Agent(生成文案) -> 审批节点(人工审核) -> 发布

**多 Agent 协作拓扑**：
- **Supervisor（主管）**：条件路由函数扮演 Supervisor，决定下一步去哪个节点
- **层级式（Hierarchical）**：可扩展为子团队（品牌团队/效果团队各含多个 Agent）
- **A2A 协议**：Google 2024 提出的 Agent-to-Agent 通信协议，与 MCP 互补（MCP 接工具，A2A 接 Agent）


## TODO 1：定义 CampaignState（企业营销编排全局状态）

**State 驱动设计**：所有节点共享全局 State，每个节点只更新自己负责的字段。
用 `TypedDict` 定义（LangGraph 推荐），`messages` 字段用 `Annotated[list, operator.add]` 实现追加模式。

需要包含：brief(输入) / audience_analysis / campaign_strategy / copy_content / approved(审批决定) / review_feedback / revision_count(循环退出用) / final_output / messages(追加)


In [ ]:
# TODO: 你的代码
# 1. 定义 CampaignState（TypedDict，含 9 个字段）
# 字段：brief, audience_analysis, campaign_strategy, copy_content,
#       approved, review_feedback, revision_count, final_output, messages
# 提示：messages 用 Annotated[list, operator.add] 实现追加模式

raise NotImplementedError


## TODO 2：实现 research_agent 和 strategy_agent（顺序编排 + Plan-Execute）

**节点模式**：`def agent(state: CampaignState) -> dict` -- 读 State、调 LLM、返回 State 更新。

- `research_agent`：分析受众/竞品/趋势（顺序编排第 1 步）
- `strategy_agent`：制定营销策略（**Plan-Execute 模式**的 Plan 阶段 -- 先规划再执行）

用 `llm.invoke([SystemMessage(...), HumanMessage(...)])` 调用 LLM，返回 `{"字段": response.content, "messages": [...]}`。


In [ ]:
# TODO: 你的代码
# 2. research_agent 和 strategy_agent
# research_agent(state) -> dict: 调用 llm 分析受众，返回 audience_analysis + messages
# strategy_agent(state) -> dict: 调用 llm 制定策略，返回 campaign_strategy + messages
# 提示：用 [SystemMessage(content=...), HumanMessage(content=...)] 调用 llm.invoke()

raise NotImplementedError


## TODO 3：实现 copywriter_agent（Plan-Execute 执行阶段 + 循环修改）

**Plan-Execute 的 Execute 阶段**：基于策略生成文案。若有审核反馈（`review_feedback`），针对性修改。

关键：检查 `state.get("review_feedback")` 是否存在，若有则加入 Prompt 让 LLM 据此改进。


In [ ]:
# TODO: 你的代码
# 3. copywriter_agent(state) -> dict
# 调用 llm 生成文案；若 state.get("review_feedback") 存在，加入 Prompt 据此修改
# 返回 copy_content + messages

raise NotImplementedError


## 2. 条件路由与循环退出（Supervisor 角色）

**条件路由**是 LangGraph 实现分支与循环的关键：
1. **条件函数**：`def route(state) -> Literal["publish", "revise"]` -- 读 State 决定下一节点
2. **条件边注册**：`workflow.add_conditional_edges("approval", route, {"publish": "publish", "revise": "copywriter"})`

**循环退出条件（必须）**：`revision_count >= 3` 时强制发布，防止无限循环。

**企业治理视角**：条件路由函数扮演 Supervisor 角色 -- 它拥有全局视图，决定工作流走向。
这比"让 LLM 自主决定下一步"（ReAct 模式）更可控、更适合生产环境。


## TODO 4：实现 approval_node 和 route_after_approval（审批 + 条件路由）

- `approval_node`：企业营销合规审核节点
  - HITL 模式：人工通过 `update_state` 注入 `approved=True`，此节点记录并放行
  - 自动模式：用 `revision_count` 模拟"首次需修改、修改后通过"
- `route_after_approval`：条件路由函数（Supervisor 角色），返回 `"publish"` 或 `"revise"`
  - 循环退出：`revision_count >= 3` 强制发布


In [ ]:
# TODO: 你的代码
# 4. approval_node 和 route_after_approval
# approval_node(state) -> dict:
#   - 若 state["approved"] 已为 True（HITL 注入）：记录并放行
#   - 若 revision_count >= 1（自动模式第二次）：设 approved=True
#   - 否则（自动模式首次）：approved=False，给 review_feedback
# route_after_approval(state) -> Literal["publish", "revise"]:
#   - revision_count >= 3: 强制 "publish"（循环退出）
#   - approved == True: "publish"
#   - 否则: "revise"

raise NotImplementedError


## 3. 发布节点（给定，无需填写）

`publish_node` 是工作流终点，把所有 Agent 输出汇总成最终方案。


In [ ]:
def publish_node(state: CampaignState) -> dict:
    """发布节点：汇总所有 Agent 输出，生成最终方案"""
    final_output = (
        "=" * 60 + "\n"
        "        企业营销活动方案 - 最终输出\n" +
        "=" * 60 + "\n\n"
        f"【营销Brief】\n{state['brief']}\n\n"
        f"【受众分析】\n{state.get('audience_analysis', '')}\n\n"
        f"【营销策略】\n{state.get('campaign_strategy', '')}\n\n"
        f"【创意文案】\n{state.get('copy_content', '')}\n\n"
        f"【审批记录】修改次数:{state.get('revision_count', 0)} "
        f"最终:{'通过' if state.get('approved') else '强制发布'}\n" +
        "=" * 60
    )
    return {
        "final_output": final_output,
        "messages": [f"[publish] 方案已发布 (修改次数={state.get('revision_count', 0)})"]
    }

print("publish_node 定义完成")


## TODO 5：实现 build_campaign_graph（Supervisor 拓扑装配 + HITL + 持久化）

**StateGraph 装配全流程**：
1. `StateGraph(CampaignState)` 创建图
2. `add_node` 添加 5 个节点
3. `add_edge(START, "research")` + 顺序边 research->strategy->copywriter->approval
4. `add_conditional_edges("approval", route, {...})` 条件边
5. `add_edge("publish", END)` 终止边
6. `compile(checkpointer=MemorySaver(), interrupt_before=["approval"])` 编译

**`use_hitl` 参数**：
- `True`（默认）：加 `interrupt_before=["approval"]`，审批前暂停等待人工
- `False`：不加 interrupt，approval_node 自动决策，图端到端跑通（演示修订循环）


In [ ]:
# TODO: 你的代码
# 5. build_campaign_graph(use_hitl: bool = True)
# 装配 StateGraph：
#   add_node: research, strategy, copywriter, approval, publish
#   add_edge: START->research, research->strategy, strategy->copywriter, copywriter->approval, publish->END
#   add_conditional_edges("approval", route_after_approval, {"publish":"publish", "revise":"copywriter"})
#   compile(checkpointer=MemorySaver(), interrupt_before=["approval"] if use_hitl else None)
# 返回编译后的图

raise NotImplementedError


## 4. 基础运行：修订循环演示（给定，无需填写）

`run_basic` 构建不带 interrupt 的图，端到端执行。
approval_node 逻辑：首次审核不通过（rc=0 -> reject），修改后通过（rc=1 -> approve）。
演示"生成 -> 审核 -> 修改 -> 再审核 -> 发布"的完整循环。


In [ ]:
def run_basic(brief: str):
    """基础运行：不带 interrupt，端到端执行，演示修订循环"""
    graph = build_campaign_graph(use_hitl=False)
    config = {"configurable": {"thread_id": "basic_001"}}
    initial_state = {
        "brief": brief, "audience_analysis": "", "campaign_strategy": "",
        "copy_content": "", "approved": False, "review_feedback": "",
        "revision_count": 0, "final_output": "", "messages": [],
    }
    print("=" * 60)
    print("基础运行（无 HITL，演示修订循环）")
    print("=" * 60)
    result = graph.invoke(initial_state, config=config)
    print("\n--- 节点执行顺序（从 messages 追踪）---")
    for i, msg in enumerate(result.get("messages", [])):
        print(f"  {i+1}. {msg}")
    print(f"\n--- 修订循环结果 ---")
    print(f"修改次数: {result.get('revision_count', 0)}")
    print(f"最终审批: {'通过' if result.get('approved') else '未通过'}")
    print(f"\n--- 最终输出（前200字）---")
    print(result.get("final_output", "")[:200] + "...")
    return result

brief = """
品牌：雅净（Yajing）
产品：新款烟酰胺精华液（5%浓度+泛醇舒缓）
目标：新品上市推广，3个月内品牌知名度提升20%
预算：50万元 | 渠道：小红书、抖音、天猫
"""

# 运行基础版（build_campaign_graph 需先完成 TODO 5）
try:
    result_basic = run_basic(brief)
except NotImplementedError:
    print("⚠️ 请先完成 TODO 1-5，再运行基础版")


## 5. 人机协同（HITL）：interrupt_before 三步模式

企业治理的关键能力：工作流在审批节点前**暂停**，State 被 MemorySaver 持久化；
人工审核后通过 `update_state` 注入决策，图从检查点**恢复**执行。

**三步模式**：
1. `graph.invoke(initial_state, config)` -> 执行到 approval 前暂停
2. `graph.update_state(config, {"approved": True})` -> 注入人工决策
3. `graph.invoke(None, config)` -> 恢复执行至完成

这是 LangGraph 相对其他框架的核心优势，也是企业 AI 治理的工程基础。


## TODO 6：实现 run_with_hitl（三步 HITL 运行 + 真实输出）

实现三步 HITL 运行：
1. 构建 `use_hitl=True` 的图
2. Step 1：invoke -> 暂停前打印已执行节点 + 暂停位置
3. Step 2：update_state 注入 `approved=True`
4. Step 3：invoke(None) 恢复 -> 打印最终输出 + 执行日志


In [ ]:
# TODO: 你的代码
# 6. run_with_hitl(brief: str)
# 三步 HITL 运行：
#   Step 1: graph = build_campaign_graph(use_hitl=True); state1 = graph.invoke(initial, config)
#           打印: 已执行节点(messages)、暂停位置(graph.get_state(config).next)
#   Step 2: graph.update_state(config, {"approved": True, "review_feedback": "人工审核通过"})
#           打印: 已注入审批结果
#   Step 3: final = graph.invoke(None, config)
#           打印: 最终输出(final_output)、执行日志(messages)

raise NotImplementedError


## 6. 反思与前沿

### 反思问题
1. 修订循环的退出条件（`revision_count >= 3`）设成多少合理？为什么？取消退出条件会怎样？
2. HITL 审批节点在企业治理中的意义是什么？哪些环节"必须"人机协同？
3. Supervisor 拓扑 vs 层级式拓扑，各适合什么规模的多 Agent 系统？

### 2026 前沿
- **A2A（Agent-to-Agent Protocol）**：Google 2024 提出的 Agent 间通信协议（https://github.com/google/A2A），与 MCP 互补：MCP 接工具，A2A 接 Agent。本 Day 的多 Agent 通过 State 共享通信；A2A 提供跨进程/跨组织 Agent 通信标准。
- **Plan-Execute 模式**：本 Day 的 strategy_agent(Plan) + copywriter_agent(Execute) 即此模式。
- **天道推演 × 多 Agent 仿真**：把天道推演的多路径沙盘映射为 LangGraph 的条件分支图 -- 每条决策路径是一个分支，Checkpointing 记录推演假设，反馈学习节点更新因果模型。可从"思维框架"升级为"可计算多 Agent 沙盘"。
